# CellOracle gene-regulatory network analysis

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
from pathlib import Path
import os

project_root = Path(os.environ.get("BMO_PROJECT_ROOT", ".")).resolve()
for directory in ("data/processed", "results/figures", "results/tables", "results/objects/grn"):
    (project_root / directory).mkdir(parents=True, exist_ok=True)

# 0. Import

import os
import sys

import matplotlib.pyplot as plt

import scanpy as sc
import seaborn as sns

import numpy as np
import pandas as pd

import celloracle as co
co.__version__


In [ ]:
# visualization settings
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300


In [ ]:
save_folder = "figures"
os.makedirs(save_folder, exist_ok=True)


In [ ]:
adata = sc.read_h5ad(str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene.h5ad'))


In [ ]:
adata


In [ ]:
print(f"Cell number is :{adata.shape[0]}")
print(f"Gene number is :{adata.shape[1]}")


In [ ]:
# Loading prebuilt promoter base-GRN. Version: hg19_gimmemotifsv5_fpr2
base_GRN = co.data.load_human_promoter_base_GRN()


In [ ]:
base_GRN.head()


In [ ]:
# Instantiate Oracle object
oracle = co.Oracle()


In [ ]:
print("Metadata columns :", list(adata.obs.columns))
print("Dimensional reduction: ", list(adata.obsm.keys()))


In [ ]:
adata.layers


In [ ]:
adata.X = adata.layers["data_RNA"].copy()

oracle.import_anndata_as_raw_count(adata=adata,
                                   cluster_column_name="celltype",
                                   embedding_name="X_tsne")


In [ ]:
oracle.import_TF_data(TF_info_matrix=base_GRN)


In [ ]:
# Perform PCA
oracle.perform_PCA()

# Select important PCs
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
plt.show()
print(n_comps)
n_comps = min(n_comps, 50)


In [ ]:
n_cell = oracle.adata.shape[0]
print(f"cell number is :{n_cell}")


In [ ]:
k = int(0.025*n_cell)
print(f"Auto-selected k is :{k}")


In [ ]:
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=4)


In [ ]:
# Save oracle object.
oracle.to_hdf5(str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.celloracle.oracle'))


In [ ]:
# Load file.
oracle = co.load_hdf5(str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.celloracle.oracle'))


In [ ]:
adata.obsm_keys()


In [ ]:
# Check clustering data
sc.pl.umap(oracle.adata, color="celltype")


In [ ]:
# Check clustering data
sc.pl.tsne(oracle.adata, color="celltype")


In [ ]:
%%time
# Calculate GRN for each population in "louvain_annot" clustering unit.
# This step may take some time.(~30 minutes)
links = oracle.get_links(cluster_name_for_GRN_unit="celltype", alpha=10,
                         verbose_level=10)


In [ ]:
links.links_dict.keys()


In [ ]:
# Show the contents of pallete
links.palette


In [ ]:
# Save Links object.
links.to_hdf5(file_path=str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.links.celloracle.links'))


In [ ]:
links.filter_links(p=0.001, weight="coef_abs", threshold_number=2000)


In [ ]:
plt.rcParams["figure.figsize"] = [9, 4.5]


In [ ]:
links.plot_degree_distributions(plot_model=True,
                                               #save=f"{save_folder}/degree_distribution/",
                                               )


In [ ]:
plt.rcParams["figure.figsize"] = [6, 4.5]


In [ ]:
# Calculate network scores.
links.get_network_score()


In [ ]:
links.merged_score.head()


In [ ]:
# Save Links object.
links.to_hdf5(file_path=str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.links.celloracle.links'))


In [ ]:
# You can load files with the following command.
links = co.load_hdf5(file_path=str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.links.celloracle.links'))


In [ ]:
# Check cluster name
links.cluster


In [ ]:
# Visualize top n-th genes with high scores.
links.plot_scores_as_rank(cluster="Erythroid Stage1", n_gene=30, save=f"{save_folder}/ranked_score")


In [ ]:
# Compare GRN score between two clusters
links.plot_score_comparison_2D(value="eigenvector_centrality",
                               cluster1="Erythroid Stage1", cluster2="Erythroid Stage2",
                               percentile=98,
                               save=f"{save_folder}/score_comparison")


In [ ]:
# Compare GRN score between two clusters
links.plot_score_comparison_2D(value="betweenness_centrality",
                               cluster1="Erythroid Stage1", cluster2="Erythroid Stage3",
                               percentile=98,
                               save=f"{save_folder}/score_comparison")


In [ ]:
# Compare GRN score between two clusters
links.plot_score_comparison_2D(value="eigenvector_centrality",
                               cluster1="Erythroid Stage2", cluster2="Erythroid Stage3",
                               percentile=98,
                               save=f"{save_folder}/score_comparison")


In [ ]:
# Visualize Gata2 network score dynamics
links.plot_score_per_cluster(goi="CEBPB", save=f"{save_folder}/network_score_per_gene/")


In [ ]:
cluster_name = "Erythroid Stage2"
filtered_links_df = links.filtered_links[cluster_name]
filtered_links_df.head()


In [ ]:
filtered_links_df[filtered_links_df.source == "CEBPB"]


In [ ]:
plt.rcParams["figure.figsize"] = [6, 4.5]


In [ ]:
# Plot degree_centrality
plt.subplots_adjust(left=0.15, bottom=0.3)
plt.ylim([0,0.040])
links.plot_score_discributions(values=["degree_centrality_all"],
                               method="boxplot",
                               save=f"{save_folder}",
                              )


In [ ]:
# Plot eigenvector_centrality
plt.subplots_adjust(left=0.15, bottom=0.3)
plt.ylim([0, 0.28])
links.plot_score_discributions(values=["eigenvector_centrality"],
                               method="boxplot",
                               save=f"{save_folder}")


In [ ]:
plt.subplots_adjust(left=0.15, bottom=0.3)
links.plot_network_entropy_distributions(save=f"{save_folder}")


<b><font size=5 color=pink >Prepare the perturbation model</font></b>


In [ ]:
import os
import sys

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns


In [ ]:
import celloracle as co
co.__version__


In [ ]:
#plt.rcParams["font.family"] = "arial"
plt.rcParams["figure.figsize"] = [6,6]
%config InlineBackend.figure_format = 'retina'
plt.rcParams["savefig.dpi"] = 600

%matplotlib inline


In [ ]:
# Make folder to save plots
save_folder = "figures"
os.makedirs(save_folder, exist_ok=True)


In [ ]:
oracle = co.load_hdf5(str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.celloracle.oracle'))


In [ ]:
links = co.load_hdf5(str(project_root / 'results' / 'objects' / 'grn' / 'BMO_Erythroid_allgene_data.links.celloracle.links'))


In [ ]:
links.filter_links()
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10,
                              use_cluster_specific_TFdict=True)


In [ ]:
# Check gene expression
goi = "CEBPB"
sc.pl.umap(oracle.adata, color=[goi, oracle.cluster_column_name],
                 layer="imputed_count", use_raw=False, cmap="viridis")


In [ ]:
# Check gene expression
goi = "CEBPB"
sc.pl.tsne(oracle.adata, color=[goi, oracle.cluster_column_name],
                 layer="imputed_count", use_raw=False, cmap="viridis")


In [ ]:
# Plot gene expression in histogram
sc.get.obs_df(oracle.adata, keys=[goi], layer="imputed_count").hist()
plt.show()


<b><font size=5 color=pink >CEBPB KO</font></b>


In [ ]:
# Enter perturbation conditions to simulate signal propagation after the perturbation.
oracle.simulate_shift(perturb_condition={goi: 0.0},
                      n_propagation=3)


In [ ]:
# Get transition probability
oracle.estimate_transition_prob(n_neighbors=200,
                                knn_random=True,
                                sampled_fraction=1)

# Calculate embedding
oracle.calculate_embedding_shift(sigma_corr=0.05)


In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 25
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 25
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
# n_grid = 40 is a good starting value.
n_grid = 40
oracle.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)


In [ ]:
# Search for best min_mass.
oracle.suggest_mass_thresholds(n_suggestion=12)


In [ ]:
min_mass = 0.01
oracle.calculate_mass_filter(min_mass=min_mass, plot=True)


In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale_simulation = 15
# Show quiver plot
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_simulation_flow_random_on_grid(scale=scale_simulation, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
# Plot vector field with cell cluster
fig, ax = plt.subplots(figsize=[8, 8])

oracle.plot_cluster_whole(ax=ax, s=10)
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax, show_background=False)


<b><font size=5 color=pink >CEBPB overexpression simulation</font></b>


In [ ]:
# Enter perturbation conditions to simulate signal propagation after the perturbation.
oracle.simulate_shift(perturb_condition={goi: 1.2},
                      n_propagation=3)


In [ ]:
# Get transition probability
oracle.estimate_transition_prob(n_neighbors=200,
                                knn_random=True,
                                sampled_fraction=1)

# Calculate embedding
oracle.calculate_embedding_shift(sigma_corr=0.05)


In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 25
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} OE")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
# n_grid = 40 is a good starting value.
n_grid = 40
oracle.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)


In [ ]:
# Search for best min_mass.
oracle.suggest_mass_thresholds(n_suggestion=12)


In [ ]:
min_mass = 0.01
oracle.calculate_mass_filter(min_mass=min_mass, plot=True)


In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale_simulation = 15
# Show quiver plot
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} OE")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_simulation_flow_random_on_grid(scale=scale_simulation, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()


In [ ]:
# Plot vector field with cell cluster
fig, ax = plt.subplots(figsize=[8, 8])

oracle.plot_cluster_whole(ax=ax, s=10)
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax, show_background=False)


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(8, 8))

oracle.plot_cluster_whole(ax=ax, s=10)
oracle.plot_simulation_flow_on_grid(
    scale=scale_simulation,
    ax=ax,
    show_background=False
)

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("Simulation vector field")

plt.tight_layout()

fig.savefig(
    str(project_root / "results" / "figures" / "CellOracle_vector_field.pdf"),
    format="pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()


In [ ]:
celltype_col = oracle.cluster_column_name
print(celltype_col)

print(oracle.adata.obs[celltype_col].value_counts())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

celltype_col = oracle.cluster_column_name
oracle.adata.obs[celltype_col] = oracle.adata.obs[celltype_col].astype("category")

celltype_colors = {
    "Erythroid Stage1": "#F4A3A3",
    "Erythroid Stage2": "#E64B35",
    "Erythroid Stage3": "#7A0019"
}

emb = oracle.embedding
celltypes = oracle.adata.obs[celltype_col].astype(str).values

celltype_order = [
    "Erythroid Stage1",
    "Erythroid Stage2",
    "Erythroid Stage3"
]

fig, ax = plt.subplots(figsize=(4, 4))

for ct in celltype_order:
    idx = celltypes == ct
    ax.scatter(
        emb[idx, 0],
        emb[idx, 1],
        s=10,
        c=celltype_colors[ct],
        label=ct,
        alpha=0.85,
        linewidths=0,
        rasterized=True,
        zorder=1
    )

n_collections_before = len(ax.collections)
n_patches_before = len(ax.patches)


oracle.plot_simulation_flow_on_grid(
    scale=scale_simulation,
    ax=ax,
    show_background=False
)

for col in ax.collections[n_collections_before:]:
    col.set_edgecolor("black")
    col.set_linewidth(0)
    col.set_path_effects([])
    col.set_zorder(3)

for patch in ax.patches[n_patches_before:]:
    patch.set_edgecolor("black")
    patch.set_linewidth(0)
    patch.set_path_effects([])
    patch.set_zorder(3)

ax.set_title("Simulation vector field", fontsize=14)
ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")


plt.tight_layout()

fig.savefig(
    str(project_root / "results" / "figures" / "CellOracle_vector_field_no_white_edge.pdf"),
    format="pdf",
    bbox_inches="tight",
    dpi=300
)

plt.show()


In [ ]:
import numpy as np

x = oracle.adata[:, "CEBPB"].X
if hasattr(x, "toarray"):
    x = x.toarray().flatten()
else:
    x = np.array(x).flatten()

print("min:", np.min(x))
print("25%:", np.percentile(x, 25))
print("50%:", np.percentile(x, 50))
print("75%:", np.percentile(x, 75))
print("90%:", np.percentile(x, 90))
print("95%:", np.percentile(x, 95))
print("max:", np.max(x))
